# Notebook 6: Movie Chatbot with TAG (Taxonomy)

## Features:
- ✅ Genre classification (10 custom genres)
- ✅ Web search
- ✅ Conversational memory
- ✅ Smart taxonomy-based recommendations

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results --quiet

In [1]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from ai_course_utils import load_llm_from_env, load_use_case_config, load_taxonomy_from_excel, display_config

load_dotenv()
load_dotenv('../.env')
print("✓ Imports successful")

✓ Imports successful


In [2]:
display_config()

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')


In [3]:
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"
use_case_config = load_use_case_config(use_case_file)
system_prompt = use_case_config.get("agent_prompt")

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url


## Load Taxonomy

In [4]:
taxonomy_file = "../data/Movie_Recommendation_Taxonomy_File.xlsx"
taxonomy = load_taxonomy_from_excel(taxonomy_file)

print(f"✓ Loaded {len(taxonomy)} genres")

✓ Taxonomy loaded: ../data/Movie_Recommendation_Taxonomy_File.xlsx
  Genres: 10 (Neo-Mythical, Futuristic Noir, Eco-Thriller...)
✓ Loaded 10 genres


## Define Tools

In [5]:
@tool
def search_web(query: str) -> str:
    """Search web for movie information."""
    search = GoogleSerperAPIWrapper()
    return search.run(query)

@tool
def classify_genre(query: str) -> dict:
    """Classify query into movie genre."""
    query_lower = query.lower()
    for genre_name, info in taxonomy.items():
        included = info.get('included', '').lower()
        keywords = [k.strip() for k in included.split(',') if k.strip()]
        if any(kw in query_lower for kw in keywords):
            return {
                'genre': genre_name,
                'description': info.get('description', ''),
                'guidelines': info.get('guidelines', '')
            }
    return {'genre': 'General'}

tools = [search_web, classify_genre]
print(f"✓ Tools: {[t.name for t in tools]}")

✓ Tools: ['search_web', 'classify_genre']


In [6]:
llm = load_llm_from_env()
model = llm.bind_tools(tools)

🤖 Loading LLM: openai / gpt-4.1-mini


## Build Agent with Memory

In [7]:
def call_model(state: MessagesState):
    return {"messages": [model.invoke([SystemMessage(content=system_prompt)] + state["messages"])]}

def should_continue(state: MessagesState):
    return "tools" if state["messages"][-1].tool_calls else END

workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

app = workflow.compile(checkpointer=MemorySaver())
print("✓ Agent with TAG + memory ready")

✓ Agent with TAG + memory ready


## Chat Helper Function

In [8]:
def chat(user_input: str, thread_id: str = "default") -> str:
    """Chat with the assistant (maintains memory)."""
    config = {"configurable": {"thread_id": thread_id}}
    result = None
    for event in app.stream({"messages": [HumanMessage(content=user_input)]}, config, stream_mode="values"):
        result = event
    return result["messages"][-1].content

print("🎬 Chatbot with TAG Ready!")

🎬 Chatbot with TAG Ready!


## Test Conversations

In [11]:
# Test with memory
thread = "genre_test"

Query1 = "Jason likes history and sports  and I like Bollywood movies. Can you recommend a Bollywood historical movie that also has sports theme"
print("User: {Query1}")
r1 = chat(Query1, thread_id=thread)
print(f"Bot: {r1[:200]}...")

print("\nUser: Recommend one based on that")
r2 = chat("Recommend one based on that", thread_id=thread)
print(f"Bot: {r2[:200]}...")

User: {Query1}
Bot: I previously recommended "Lagaan: Once Upon a Time in India" as a Bollywood historical sports movie. Would you like another recommendation, or more details about that one? Also, do you prefer a movie ...

User: Recommend one based on that
Bot: Another excellent Bollywood historical sports movie you might enjoy is "Bhaag Milkha Bhaag" (2013). It's a biographical film based on the life of Milkha Singh, an Indian athlete who overcame great per...


In [ ]:
# Test with memory
thread = "genre_test"
print("User: I love mythology mixed with sci-fi")
r1 = chat("I love mythology mixed with sci-fi", thread_id=thread)
print(f"Bot: {r1[:200]}...")

print("\nUser: Recommend one based on that")
r2 = chat("Recommend one based on that", thread_id=thread)
print(f"Bot: {r2[:200]}...")